# Módulo 09 · Rendimiento

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`, escaladas sintéticamente)

## Objetivos
1. **Medir** antes de optimizar: tiempo y memoria.
2. Reemplazar bucles y `apply` por operaciones **vectorizadas** (incluida lógica condicional).
3. Entender qué cambia **Copy-on-Write** en memoria y por qué `.copy()` casi nunca hace falta.
4. Elegir **dtypes** que aceleran y ahorran memoria (`category`, *downcast*, Arrow).
5. Usar **numba** cuando necesitas una función propia y no hay alternativa vectorizada.
6. Leer y escribir eficientemente: **Parquet** (columnas, filtros, particiones) y **lectura por chunks**.

| Sección | Tema |
|---|---|
| 0 | Setup: un dataset grande |
| 1 | Medir |
| 2 | Vectorización |
| 3 | Lógica condicional vectorizada |
| 4 | Lookups rápidos |
| 5 | Copy-on-Write y memoria |
| 6 | Dtypes para rendimiento |
| 7 | numba |
| 8 | I/O: Parquet y chunks |
| 9 | Checklist |
| ✍️ | Ejercicios · ✅ Soluciones |

> ⏱️ Los tiempos que verás dependen de tu equipo. Fíjate en las **proporciones** (10×, 100×), no en los milisegundos exactos.

## 0 · Setup: un dataset grande

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
TMP = Path("cache") / "m09"
TMP.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


def mb(df) -> float:
  return round(df.memory_usage(deep=True).sum() / 1024**2, 2)


print("pandas", pd.__version__)

Con 336 ítems todo es instantáneo. Para que las diferencias se vean, **escalamos** la muestra a ~1 millón de filas, replicándola con ruido aleatorio en precios y fechas. Son datos **sintéticos** derivados de la muestra: sirven para medir, no para sacar conclusiones de negocio.

In [ ]:
items = pd.read_csv(DATA / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"],
                     usecols=["order_id", "customer_id", "order_status", "order_purchase_timestamp"])
customers = pd.read_csv(DATA / "olist_customers_dataset.csv", usecols=["customer_id", "customer_state"])
products = pd.read_csv(DATA / "olist_products_dataset.csv", usecols=["product_id", "product_category_name"])

base = (items
        .merge(orders, on="order_id", validate="m:1")
        .merge(customers, on="customer_id", validate="m:1")
        .merge(products, on="product_id", validate="m:1")
        .drop(columns=["customer_id", "shipping_limit_date"]))

rng = np.random.default_rng(42)
REPS = 3000
grande = pd.concat([base] * REPS, ignore_index=True)
n = len(grande)
grande = grande.assign(
  price=(grande["price"] * rng.uniform(0.8, 1.2, n)).round(2),
  freight_value=(grande["freight_value"] * rng.uniform(0.8, 1.2, n)).round(2),
  order_purchase_timestamp=grande["order_purchase_timestamp"] + pd.to_timedelta(rng.integers(-180, 180, n), unit="D"),
  order_id=grande["order_id"] + "_" + (np.arange(n) // len(base)).astype(str),
)
print(f"{len(grande):,} filas | {mb(grande)} MB")
grande.head(3)

## 1 · Medir

- `%timeit expr`: repite la expresión muchas veces y da media ± desviación. Para operaciones rápidas.
- `%%time` (en la primera línea de una celda): mide **una** ejecución. Para operaciones largas.
- `df.memory_usage(deep=True)`: memoria real por columna (Módulo 01).

Una función propia para comparar varias alternativas en una tabla:

In [ ]:
def comparar(**alternativas) -> pd.DataFrame:
  # Ejecuta cada función 3 veces y reporta el mejor tiempo
  filas = {}
  for nombre, f in alternativas.items():
    tiempos = []
    for _ in range(3):
      t0 = time.perf_counter()
      f()
      tiempos.append(time.perf_counter() - t0)
    filas[nombre] = min(tiempos) * 1000
  res = pd.DataFrame({"ms": filas}).sort_values("ms")
  return res.assign(veces_mas_lento=lambda d: (d["ms"] / d["ms"].min()).round(1)).round(2)

## 2 · Vectorización

Calculemos el total `price + freight_value` de cinco formas sobre una porción de 20 mil filas (los bucles serían demasiado lentos sobre el millón):

In [ ]:
muestra = grande.iloc[:20_000]


def con_iterrows():
  return pd.Series([fila["price"] + fila["freight_value"] for _, fila in muestra.iterrows()])

def con_itertuples():
  return pd.Series([f.price + f.freight_value for f in muestra.itertuples(index=False)])

def con_apply():
  return muestra.apply(lambda f: f["price"] + f["freight_value"], axis=1)

def con_zip():
  return pd.Series([p + f for p, f in zip(muestra["price"], muestra["freight_value"])])

def vectorizado():
  return muestra["price"] + muestra["freight_value"]

def numpy_puro():
  return muestra["price"].to_numpy() + muestra["freight_value"].to_numpy()


comparar(iterrows=con_iterrows, itertuples=con_itertuples, apply_axis1=con_apply,
         zip=con_zip, vectorizado=vectorizado, numpy=numpy_puro)

Lectura:

- `iterrows` crea una Series **por fila**: es lo más lento, por órdenes de magnitud.
- `apply(axis=1)` es, en el fondo, un bucle de Python: casi igual de lento.
- `itertuples`/`zip` son bucles más ligeros, pero siguen en Python.
- La operación vectorizada corre en C sobre arreglos completos.

> 🎯 **Regla:** si escribes `for` o `apply(axis=1)` sobre filas de un DataFrame, busca primero la versión vectorizada.

## 3 · Lógica condicional vectorizada

El `if/elif/else` fila a fila es el caso más común de `apply` innecesario.

| Necesito | Vectorizado |
|---|---|
| 2 ramas | `np.where(cond, a, b)` o `Series.where` / `mask` |
| varias ramas | `np.select([conds], [valores], default=)` |
| rangos numéricos | `pd.cut` (cortes fijos) / `pd.qcut` (cuantiles) |
| mapear valores | `Series.map(dict)` |

In [ ]:
def segmento_apply(fila):
  if fila["price"] >= 500:
    return "premium"
  elif fila["price"] >= 100:
    return "medio"
  elif fila["freight_value"] > fila["price"]:
    return "flete_caro"
  else:
    return "basico"


m = grande.iloc[:50_000]

def con_select():
  return np.select(
    [m["price"] >= 500, m["price"] >= 100, m["freight_value"] > m["price"]],
    ["premium", "medio", "flete_caro"],
    default="basico",
  )

assert (m.apply(segmento_apply, axis=1).to_numpy() == con_select()).all()
comparar(apply=lambda: m.apply(segmento_apply, axis=1), np_select=con_select)

In [ ]:
# Rangos con pd.cut: además devuelve una categoría ORDENADA
pd.cut(grande["price"], bins=[0, 50, 100, 500, np.inf], right=False,
       labels=["<50", "50-100", "100-500", "500+"]).value_counts().sort_index()

## 4 · Lookups rápidos

Traer un valor de una tabla de referencia (estado → región). Tres formas:

In [ ]:
REGION = {"SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste", "ES": "Sudeste",
          "PR": "Sul", "SC": "Sul", "RS": "Sul",
          "BA": "Nordeste", "PE": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "AL": "Nordeste",
          "SE": "Nordeste", "RN": "Nordeste", "PB": "Nordeste", "PI": "Nordeste",
          "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
          "PA": "Norte", "AM": "Norte", "TO": "Norte", "RO": "Norte", "AC": "Norte", "AP": "Norte", "RR": "Norte"}
ref = pd.DataFrame({"customer_state": list(REGION), "region": list(REGION.values())})
estado = grande["customer_state"]
estado_cat = estado.astype("category")

comparar(
  apply_dict=lambda: estado.apply(lambda e: REGION[e]),
  map_dict=lambda: estado.map(REGION),
  merge=lambda: grande[["customer_state"]].merge(ref, on="customer_state", how="left"),
  map_sobre_category=lambda: estado_cat.map(REGION),
)

`map` sobre una columna `category` sólo traduce las **categorías** (27 valores), no el millón de filas: por eso es mucho más rápido. La misma idea que en los Módulos 01 y 07.

## 5 · Copy-on-Write y memoria

Con CoW (siempre activo en pandas 3), las operaciones que **seleccionan** o **renombran** devuelven objetos que **comparten memoria** con el original hasta que alguien escribe. La copia ocurre sólo entonces, y sólo de lo que se modifica.

In [ ]:
sub = grande[["price", "freight_value"]]
renombrado = grande.rename(columns={"price": "precio"})

print("¿Selección comparte memoria?   ", np.shares_memory(sub["price"].to_numpy(), grande["price"].to_numpy()))
print("¿Renombrado comparte memoria?  ", np.shares_memory(renombrado["precio"].to_numpy(), grande["price"].to_numpy()))

sub.loc[0, "price"] = -1        # al escribir, se copia
print("¿Tras escribir comparte?       ", np.shares_memory(sub["price"].to_numpy(), grande["price"].to_numpy()))
print("Original intacto:", grande.loc[0, "price"] != -1)

Consecuencias prácticas:

- **No pongas `.copy()` "por si acaso"** después de seleccionar o filtrar: con CoW ya es seguro modificar el resultado sin afectar el original, y la copia defensiva gasta memoria.
- Las cadenas de métodos (Módulo 08) son baratas: cada paso que no modifica datos no copia.
- `.copy()` sigue teniendo sentido cuando quieres **liberar** un DataFrame grande del que sólo te quedas con una parte pequeña (para que el original pueda borrarse de memoria).

In [ ]:
comparar(
  rename_cow=lambda: grande.rename(columns={"price": "precio"}),
  rename_y_copy=lambda: grande.rename(columns={"price": "precio"}).copy(),
)

## 6 · Dtypes para rendimiento

### Memoria

In [ ]:
pd.DataFrame({"dtype": grande.dtypes.astype(str),
              "MB": grande.memory_usage(deep=True, index=False).div(1024**2).round(1),
              "unicos": grande.nunique()})

In [ ]:
optimizado = grande.astype({
  "order_status": "category",
  "customer_state": "category",
  "product_category_name": "category",
  "seller_id": "category",
  "product_id": "category",
  "order_item_id": "int8",
}).assign(
  price=lambda d: pd.to_numeric(d["price"], downcast="float"),
  freight_value=lambda d: pd.to_numeric(d["freight_value"], downcast="float"),
)
print("Antes  :", mb(grande), "MB")
print("Después:", mb(optimizado), "MB")

> ⚠️ `float32` tiene ~7 dígitos significativos. Para montos de dinero que vas a **sumar** en grandes cantidades, el error de redondeo puede notarse; para métricas aproximadas o modelos, suele estar bien.

In [ ]:
print(f"Suma float64: {grande['price'].sum():,.2f}")
print(f"Suma float32: {optimizado['price'].sum():,.2f}")

### Velocidad: `groupby` con llaves `category` vs `str`

In [ ]:
comparar(
  groupby_str=lambda: grande.groupby("product_category_name")["price"].sum(),
  groupby_category=lambda: optimizado.groupby("product_category_name", observed=True)["price"].sum(),
)

## 7 · numba

Cuando necesitas una **función propia** que no existe vectorizada (por ejemplo, una métrica de ventana poco común), `numba` compila tu función Python a código máquina. pandas lo integra con `engine="numba"` en `rolling().apply`, `groupby().agg/transform` y otros.

> La **primera** llamada incluye el tiempo de compilación; se nota la ganancia a partir de la segunda.

In [ ]:
serie_diaria = (grande.set_index("order_purchase_timestamp")["price"]
                       .resample("h").sum())
print(f"{len(serie_diaria):,} horas")


def rango_relativo(x):
  # (máximo - mínimo) / media de la ventana
  m = x.mean()
  return (x.max() - x.min()) / m if m != 0 else np.nan


serie_diaria.rolling(48).apply(rango_relativo, raw=True, engine="numba")   # compila (primera vez)

comparar(
  python=lambda: serie_diaria.rolling(48).apply(rango_relativo, raw=True),
  numba=lambda: serie_diaria.rolling(48).apply(rango_relativo, raw=True, engine="numba"),
)

`raw=True` pasa arreglos NumPy (no Series) a la función: necesario para numba y más rápido incluso sin él.

Con `groupby`, la función recibe `(values, index)`:

In [ ]:
def rango_relativo_grupo(values, index):
  m = values.mean()
  return (values.max() - values.min()) / m if m != 0 else np.nan


g = optimizado.groupby("seller_id", observed=True)["price"]
g.agg(rango_relativo_grupo, engine="numba")      # compila

comparar(
  python=lambda: g.agg(lambda s: (s.max() - s.min()) / s.mean()),
  numba=lambda: g.agg(rango_relativo_grupo, engine="numba"),
  vectorizado=lambda: (g.max() - g.min()) / g.mean(),
)

Observa que la versión **vectorizada con métodos built-in** suele ganar o empatar con numba. Orden de preferencia:

1. Métodos built-in vectorizados.
2. Combinaciones de ellos (como arriba).
3. numba para lo que no se puede expresar así.
4. `apply` con Python puro, sólo para datos pequeños.

## 8 · I/O: Parquet y lectura por chunks

### CSV vs Parquet

In [ ]:
csv_path = TMP / "grande.csv"
pq_path = TMP / "grande.parquet"

t0 = time.perf_counter(); optimizado.to_csv(csv_path, index=False); t_csv_w = time.perf_counter() - t0
t0 = time.perf_counter(); optimizado.to_parquet(pq_path, index=False); t_pq_w = time.perf_counter() - t0

t0 = time.perf_counter(); pd.read_csv(csv_path); t_csv_r = time.perf_counter() - t0
t0 = time.perf_counter(); pd.read_csv(csv_path, engine="pyarrow"); t_csv_arrow = time.perf_counter() - t0
t0 = time.perf_counter(); pd.read_parquet(pq_path); t_pq_r = time.perf_counter() - t0
t0 = time.perf_counter(); pd.read_parquet(pq_path, columns=["seller_id", "price"]); t_pq_cols = time.perf_counter() - t0

pd.DataFrame({
  "tamaño_MB": [csv_path.stat().st_size / 1024**2, None, pq_path.stat().st_size / 1024**2, None],
  "escribir_s": [t_csv_w, None, t_pq_w, None],
  "leer_s": [t_csv_r, t_csv_arrow, t_pq_r, t_pq_cols],
}, index=["CSV", "CSV (engine=pyarrow)", "Parquet", "Parquet (2 columnas)"]).round(2)

Parquet es columnar y comprimido: archivos más pequeños, lectura mucho más rápida y, sobre todo, **conserva los dtypes** (categorías, fechas). Leer sólo las columnas que necesitas es casi gratis.

### Particiones y filtros

`partition_cols` escribe una carpeta por valor (p. ej. `anio=2018/`). Al leer con `filters`, pyarrow **no abre** las particiones que no cumplen la condición.

In [ ]:
part_path = TMP / "particionado"
(optimizado
 .assign(anio=lambda d: d["order_purchase_timestamp"].dt.year)
 .to_parquet(part_path, partition_cols=["anio"], index=False))

sorted(p.name for p in part_path.iterdir())

In [ ]:
solo_2018 = pd.read_parquet(part_path, filters=[("anio", "=", 2018)], columns=["seller_id", "price", "anio"])
print(f"{len(solo_2018):,} filas de 2018")
comparar(
  leer_todo_y_filtrar=lambda: pd.read_parquet(part_path).loc[lambda d: d["anio"] == 2018],
  leer_con_filtro=lambda: pd.read_parquet(part_path, filters=[("anio", "=", 2018)], columns=["seller_id", "price"]),
)

### Lectura por chunks

Cuando un CSV **no cabe en memoria**, léelo por partes con `chunksize` y acumula **resultados parciales** (sumas, conteos), no los datos.

In [ ]:
parciales = []
for chunk in pd.read_csv(csv_path, usecols=["customer_state", "price"], chunksize=200_000):
  parciales.append(chunk.groupby("customer_state")["price"].agg(["sum", "count"]))

por_estado = pd.concat(parciales).groupby(level=0).sum()
por_estado = por_estado.assign(media=lambda d: d["sum"] / d["count"]).sort_values("sum", ascending=False)
por_estado.head()

> ⚠️ No todas las métricas se combinan así: la **suma** y el **conteo** sí; la **media** se reconstruye como suma/conteo; pero la **mediana** o `nunique` no se pueden combinar a partir de resultados parciales. Para esas, usa Parquet + columnas, o herramientas como DuckDB/Polars.

## 9 · Checklist de rendimiento

1. **Mide** primero (`%timeit`, `memory_usage(deep=True)`). Optimiza lo que realmente tarda.
2. **Lee menos**: `usecols`/`columns`, `filters`, `dtype` al leer, Parquet en lugar de CSV.
3. **Dtypes**: `category` para baja cardinalidad, enteros pequeños, `str` (Arrow) para texto.
4. **Vectoriza**: nada de `iterrows`/`apply(axis=1)`; `np.select`, `pd.cut`, `map`.
5. **Agrega antes de unir** (Módulo 05) y filtra lo antes posible.
6. **Built-ins** en `groupby` (`"sum"`, `"mean"`), columnas auxiliares antes de agrupar.
7. **numba** sólo para funciones propias sin alternativa vectorizada.
8. **CoW**: sin `.copy()` defensivos; encadena sin miedo.
9. Si aun así no alcanza: chunks, o herramientas para datos más grandes que la memoria.

---
# ✍️ Ejercicios

Variables y funciones disponibles: `grande`, `optimizado`, `comparar`, `mb`, `REGION`, `csv_path`, `pq_path`, `part_path`, `TMP`.

### Ejercicio 1 · Vectorizar una regla de negocio ⭐

Esta función clasifica cada ítem con una regla de envío. Escribe `clasificar_vec(df)` que devuelva **exactamente** lo mismo (como array o Series) sin bucles ni `apply`:

```python
def clasificar_lento(df):
    resultado = []
    for _, f in df.iterrows():
        if f["freight_value"] == 0:
            resultado.append("gratis")
        elif f["freight_value"] / f["price"] > 0.5:
            resultado.append("caro")
        elif f["customer_state"] in ("SP", "RJ", "MG") and f["price"] >= 100:
            resultado.append("prioritario")
        else:
            resultado.append("estandar")
    return pd.Series(resultado, index=df.index)
```

Después, `aceleracion`: cuántas veces más rápida es tu versión sobre `grande.iloc[:20_000]` (usa `comparar`).

In [ ]:
def clasificar_lento(df):
  resultado = []
  for _, f in df.iterrows():
    if f["freight_value"] == 0:
      resultado.append("gratis")
    elif f["freight_value"] / f["price"] > 0.5:
      resultado.append("caro")
    elif f["customer_state"] in ("SP", "RJ", "MG") and f["price"] >= 100:
      resultado.append("prioritario")
    else:
      resultado.append("estandar")
  return pd.Series(resultado, index=df.index)


# TU CÓDIGO AQUÍ
def clasificar_vec(df):
  ...

aceleracion = ...

In [ ]:
# Comprobación
if pendiente(aceleracion):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  _m = grande.iloc[:20_000]
  assert (np.asarray(clasificar_vec(_m)) == clasificar_lento(_m).to_numpy()).all(), "Los resultados no coinciden"
  assert aceleracion > 20, f"aceleracion = {aceleracion}: ¿seguro que no hay bucles?"
  assert pd.Series(np.asarray(clasificar_vec(grande))).value_counts().to_dict() == {"estandar": 601129, "prioritario": 238791, "caro": 162080, "gratis": 6000}
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Optimizador de memoria automático ⭐⭐

Escribe `optimizar(df, umbral_cat=0.05)` que devuelva un DataFrame nuevo donde:

- columnas de texto con proporción de valores únicos (`nunique / len`) **menor** a `umbral_cat` → `category`;
- enteros → el entero más pequeño posible (`pd.to_numeric(..., downcast="integer")`);
- flotantes → **se quedan** en `float64` (por la advertencia del dinero);
- el resto, sin cambios.

Aplícalo a `grande` y guarda `opt2`, `ahorro_pct` (% de memoria ahorrada) y `columnas_cat` (lista ordenada de columnas que quedaron como `category`).

In [ ]:
# TU CÓDIGO AQUÍ
def optimizar(df, umbral_cat=0.05):
  ...

opt2 = ...
ahorro_pct = ...
columnas_cat = ...

In [ ]:
# Comprobación
if pendiente(opt2, ahorro_pct, columnas_cat):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert opt2 is not grande and opt2.shape == grande.shape
  assert columnas_cat == ["customer_state", "order_status", "product_category_name", "product_id", "seller_id"], columnas_cat
  assert str(opt2["order_item_id"].dtype) == "int8"
  assert str(opt2["price"].dtype) == "float64"
  assert (opt2["price"] == grande["price"]).all()
  assert (opt2["customer_state"].astype(str) == grande["customer_state"]).all()
  assert ahorro_pct > 50, ahorro_pct
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Agregación por chunks ⭐⭐

Usando **sólo** lectura por chunks de `csv_path` (`chunksize=150_000`), calcula por `seller_id`:

- `ingreso` (suma de `price`), `items` (conteo) y `precio_max` (máximo).

Guarda el resultado en `por_vendedor_chunks` (índice `seller_id`, ordenado por índice). Verifica contra el cálculo en memoria sobre `optimizado` (la comprobación lo hace por ti).

In [ ]:
# TU CÓDIGO AQUÍ
por_vendedor_chunks = ...

In [ ]:
# Comprobación
if pendiente(por_vendedor_chunks):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  ref = (optimizado.groupby("seller_id", observed=True)
                    .agg(ingreso=("price", "sum"), items=("price", "size"), precio_max=("price", "max"))
                    .sort_index())
  ref.index = ref.index.astype(str)
  assert list(por_vendedor_chunks.columns) == ["ingreso", "items", "precio_max"]
  assert por_vendedor_chunks.index.equals(ref.index)
  assert (por_vendedor_chunks["items"].to_numpy() == ref["items"].to_numpy()).all()
  assert np.allclose(por_vendedor_chunks["ingreso"], ref["ingreso"], rtol=1e-4)
  assert np.allclose(por_vendedor_chunks["precio_max"], ref["precio_max"], rtol=1e-6)
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · numba para una métrica propia ⭐⭐

Queremos, por categoría, la **proporción de ítems cuyo precio supera 1.5 veces la mediana** de su categoría (detección simple de precios atípicos).

1. `atipicos_numba`: hazlo con `groupby(...).agg(func, engine="numba")`, donde `func(values, index)` usa NumPy dentro.
2. `atipicos_vec`: hazlo **sin** numba ni funciones propias: con `transform("median")` y una columna auxiliar.
3. Ambos deben coincidir. ¿Cuál es más rápido? Guarda la comparación en `tabla_e4` (usa `comparar`).

Usa `optimizado` y la columna `product_category_name`.

In [ ]:
# TU CÓDIGO AQUÍ
atipicos_numba = ...
atipicos_vec = ...
tabla_e4 = ...

In [ ]:
# Comprobación
if pendiente(atipicos_numba, atipicos_vec, tabla_e4):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert np.allclose(atipicos_numba.sort_index(), atipicos_vec.sort_index())
  assert len(atipicos_vec) == optimizado["product_category_name"].nunique()
  assert set(tabla_e4.index) >= {"numba", "vectorizado"}
  assert np.isclose(atipicos_vec.max(), 0.5, atol=1e-4), atipicos_vec.max()
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: dataset particionado y consulta eficiente ⭐⭐⭐

1. Escribe `optimizado` en `TMP / "por_estado_mes"` como Parquet particionado por `customer_state` y `mes` (string `"AAAA-MM"` del `order_purchase_timestamp`).
2. `n_particiones`: número de carpetas **hoja** creadas (combinaciones estado × mes).
3. `consulta`: leyendo con `filters` y `columns` (sin cargar todo), el ingreso total (`price`) y el número de ítems por `mes` para **SP y RJ** en **el primer trimestre de 2018**. DataFrame indexado por `mes` con columnas `ingreso` e `items`.
4. Verifica (la comprobación lo hace) que coincide con el cálculo sobre `optimizado` en memoria.

Pista: los filtros aceptan `("columna", "in", [...])`.

In [ ]:
# TU CÓDIGO AQUÍ
n_particiones = ...
consulta = ...

In [ ]:
# Comprobación
if pendiente(n_particiones, consulta):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert n_particiones == 474, n_particiones
  assert list(consulta.columns) == ["ingreso", "items"]
  assert list(consulta.index.astype(str)) == ["2018-01", "2018-02", "2018-03"]
  _ref = (optimizado
          .assign(mes=lambda d: d["order_purchase_timestamp"].dt.strftime("%Y-%m"))
          .query("customer_state in ['SP', 'RJ'] and mes in ['2018-01', '2018-02', '2018-03']")
          .groupby("mes")["price"].agg(["sum", "size"]))
  assert np.allclose(consulta["ingreso"].to_numpy(), _ref["sum"].to_numpy(), rtol=1e-5)
  assert (consulta["items"].to_numpy() == _ref["size"].to_numpy()).all()
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
def clasificar_vec(df):
  return np.select(
    [
      df["freight_value"] == 0,
      df["freight_value"] / df["price"] > 0.5,
      df["customer_state"].isin(["SP", "RJ", "MG"]) & (df["price"] >= 100),
    ],
    ["gratis", "caro", "prioritario"],
    default="estandar",
  )


_m = grande.iloc[:20_000]
tabla_e1 = comparar(lento=lambda: clasificar_lento(_m), vectorizado=lambda: clasificar_vec(_m))
aceleracion = tabla_e1.loc["lento", "veces_mas_lento"]
tabla_e1

*Nota:* `np.select` respeta el **orden** de las condiciones igual que el `if/elif`: un ítem con flete 0 nunca llega a evaluarse como "caro". Si la división `freight/price` pudiera dividir entre 0, NumPy daría `inf` (no error), y como la primera condición ya cubre `freight == 0`, el resultado sigue siendo correcto.

### Solución 2

In [ ]:
def optimizar(df, umbral_cat=0.05):
  cambios = {}
  for col in df.columns:
    s = df[col]
    if pd.api.types.is_string_dtype(s) and s.nunique() / len(s) < umbral_cat:
      cambios[col] = s.astype("category")
    elif pd.api.types.is_integer_dtype(s):
      cambios[col] = pd.to_numeric(s, downcast="integer")
  return df.assign(**cambios)


opt2 = optimizar(grande)
ahorro_pct = (1 - mb(opt2) / mb(grande)) * 100
columnas_cat = sorted(c for c in opt2.columns if isinstance(opt2[c].dtype, pd.CategoricalDtype))
round(ahorro_pct, 1), columnas_cat

*Nota:* `order_id` no se convierte: con el escalado casi todos sus valores son únicos, y como categoría ocuparía **más**. El umbral de cardinalidad es justo lo que evita ese error.

### Solución 3

In [ ]:
parciales = []
for chunk in pd.read_csv(csv_path, usecols=["seller_id", "price"], chunksize=150_000):
  parciales.append(chunk.groupby("seller_id")["price"].agg(["sum", "size", "max"]))

por_vendedor_chunks = (
  pd.concat(parciales)
    .groupby(level=0)
    .agg(ingreso=("sum", "sum"), items=("size", "sum"), precio_max=("max", "max"))
    .sort_index()
)
por_vendedor_chunks.head()

*Nota:* cada métrica se combina con su operación "natural": las sumas y conteos se **suman**, los máximos se combinan con **max**. La tolerancia en la comprobación se debe a que el CSV se escribió desde columnas `float32`.

### Solución 4

In [ ]:
def frac_atipicos(values, index):
  med = np.median(values)
  return (values > 1.5 * med).mean()


g4 = optimizado.groupby("product_category_name", observed=True)["price"]
atipicos_numba = g4.agg(frac_atipicos, engine="numba")

atipicos_vec = (
  optimizado
  .assign(atipico=lambda d: d["price"] > 1.5 * g4.transform("median"))
  .groupby("product_category_name", observed=True)["atipico"].mean()
)

tabla_e4 = comparar(
  numba=lambda: g4.agg(frac_atipicos, engine="numba"),
  vectorizado=lambda: optimizado.assign(atipico=lambda d: d["price"] > 1.5 * g4.transform("median"))
                                .groupby("product_category_name", observed=True)["atipico"].mean(),
)
tabla_e4

*Nota:* aquí la diferencia es pequeña y, según tu equipo, puede ganar cualquiera de las dos: la versión vectorizada hace dos pasadas (`transform` + `mean`) y crea una columna auxiliar, mientras que numba lo resuelve en una pasada por grupo. Ambas son mucho más rápidas que un `apply` con Python. numba brilla sobre todo cuando la lógica **no** se puede descomponer en built-ins (bucles con estado, condiciones que dependen del valor anterior, etc.).

### Solución 5

In [ ]:
destino = TMP / "por_estado_mes"
(optimizado
 .assign(mes=lambda d: d["order_purchase_timestamp"].dt.strftime("%Y-%m"),
         customer_state=lambda d: d["customer_state"].astype(str))
 .to_parquet(destino, partition_cols=["customer_state", "mes"], index=False, existing_data_behavior="delete_matching"))

n_particiones = sum(1 for p in destino.glob("*/*") if p.is_dir())

consulta = (
  pd.read_parquet(
    destino,
    columns=["price", "mes"],
    filters=[("customer_state", "in", ["SP", "RJ"]),
             ("mes", "in", ["2018-01", "2018-02", "2018-03"])],
  )
  .assign(mes=lambda d: d["mes"].astype(str))
  .groupby("mes")
  .agg(ingreso=("price", "sum"), items=("price", "size"))
)
n_particiones, consulta

*Notas:*

- Al leer un dataset particionado, las columnas de partición vuelven como `category`; por eso convertimos `mes` a texto antes de agrupar.
- `existing_data_behavior="delete_matching"` permite **re-ejecutar** la celda sin duplicar archivos en las particiones.
- Elegir particiones es un equilibrio: demasiado finas (miles de archivos pequeños) también son lentas. Particiona por las columnas por las que **filtras** con frecuencia y con cardinalidad moderada.

---
## Resumen del módulo

- Mide primero; compara proporciones, no milisegundos.
- `iterrows` y `apply(axis=1)` son bucles de Python: vectoriza con operaciones de columnas, `np.select`, `pd.cut`, `map`.
- `map` sobre `category` traduce categorías, no filas.
- Copy-on-Write: selecciones y renombres comparten memoria; no hagas `.copy()` defensivos.
- Dtypes: `category`, enteros pequeños; cuidado con `float32` para dinero.
- numba (`engine="numba"`) para funciones propias sin alternativa built-in.
- Parquet: más pequeño, más rápido, conserva tipos; `columns` + `filters` + particiones.
- Chunks: acumula resultados **combinables** (sumas, conteos, máximos).

**Siguiente:** Módulo 10 · Extender pandas: accessors propios, `Styler` y proyecto final integrador.